# Лабораторная работа 06. Scikit-learn: сравнение моделей и итоговая проверка

**Курс:** «Анализ данных и искусственный интеллект», 1 курс бакалавриата.

В работе 05 мы научились создавать модель, обучать её и читать ошибки. Теперь меняем один элемент за раз: способ оценки цены или параметр модели. Данные и части сохраняем, чтобы сравнение было понятным.

**После работы:** вы сможете собрать `Pipeline`, сравнить линейную модель, соседей, дерево и лес, объяснить гиперпараметр, прочитать кросс-валидацию и проверить выбранную модель на отложенном тесте.

**Основной маршрут:** разделы 1–8. Раздел 7 нужен для завершения эксперимента; после него параметры больше не подбираются. Разделы 9–12 — дополнительный каталог, их стоит пройти **перед** итоговым выбором, если хотите включить их в сравнение.

**Спокойный темп:** разделы 1–4 — первая часть занятия; сравнение, кросс-валидация и итоговая проверка — вторая. Для подробного разбора можно выделить две пары. Дополнительные темы не нужно осваивать одновременно с первым деревом.


## 1. Восстанавливаем те же данные и части

Начинаем с нового ядра: не используем переменные и обученные модели из 05. Положите тот же `cars_processed.csv` рядом с ноутбуком. Импортируем знакомые инструменты.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import sklearn

print('scikit-learn:', sklearn.__version__)


Подключение библиотеки не использует данные. Ни одной модели пока нет.


### Загружаем таблицу

Берём четыре численных признака. Технические номера строк, средние цены групп и другие случайные колонки игнорируем.


In [ ]:
data_path = Path('cars_processed.csv')
if data_path.exists():
    df = pd.read_csv(data_path)
else:
    df = pd.read_csv(
        'https://raw.githubusercontent.com/MVRonkin/BasicDataAnalysisCourse/'
        'refs/heads/main/Workshops/cars_processed.csv'
    )
df = df.rename(columns={'Price(euro)': 'Price', 'Engine_capacity(cm3)': 'Engine'})
df = df.drop(columns=[c for c in df.columns if c.startswith('Unnamed:')])
df = df.reset_index(drop=True)
NUMERIC_FEATURES = ['Year', 'Distance', 'Engine', 'km_year']
if df[NUMERIC_FEATURES + ['Price']].isna().any().any():
    raise ValueError('В выбранных колонках есть пропуски: вернитесь к подготовке данных.')
if not np.isfinite(df[NUMERIC_FEATURES + ['Price']].to_numpy()).all():
    raise ValueError('В выбранных колонках есть бесконечные значения.')
if not (df['Price'] > 0).all():
    raise ValueError('Для этого учебного примера нужны положительные цены.')
df.head(3)


Проверяем, что загружен тот же файл, что и в 05. Цена остаётся в евро; никаких преобразований целевой переменной пока нет.


### Повторяем разделение 60/20/20

Случайные состояния и порядок строк такие же, как в 05. Валидация используется для выбора, тест остаётся закрытым.


In [ ]:
X = df[NUMERIC_FEATURES].copy()
y = df['Price'].copy()

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.4, random_state=42
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_holdout, y_holdout, test_size=0.5, random_state=43
)

print('Train:', X_train.shape, 'Valid:', X_valid.shape, 'Test:', X_test.shape)


Работа 06 запускается независимо от 05, но воспроизводит её части. Для другого входного файла совпадение не гарантируется.

**Остановитесь:** по какой части можно выбирать глубину дерева? По какой нельзя?


## 2. Pipeline: соединяем преобразование и модель

В 05 мы отдельно обучили scaler, преобразовали признаки, обучили регрессию и сделали прогноз. `Pipeline` хранит эту последовательность в одном объекте.

| Ручная последовательность | Внутри Pipeline |
|---|---|
| Scaler: fit на train, затем transform | Первый шаг получает обучающие строки |
| Модель: fit на преобразованных train | Последний шаг учится на преобразованных признаках |
| Scaler: transform новой строки | При predict преобразование выполняется автоматически |
| Модель: predict | Возвращается оценка цены |

Названия шагов задаём сами: `'scale'` и `'model'`. Это имена для обращения к шагам, а не специальные команды Python.


### Создаём последовательность

`Pipeline` получает список пар `(имя, объект)`. Список читается сверху вниз. Создание последовательности пока не обучает её.


In [ ]:
from sklearn.pipeline import Pipeline

linear_pipeline = Pipeline([
    ('scale', StandardScaler()),
    ('model', LinearRegression())
])
linear_pipeline


В объекте два шага. Ни средние scaler, ни коэффициенты регрессии пока не вычислены.


### Обучаем Pipeline

Передаём обычную таблицу с исходными единицами. Pipeline сама масштабирует её по обучающим параметрам.


In [ ]:
linear_pipeline.fit(X_train, y_train)
linear_valid_pred = linear_pipeline.predict(X_valid)
linear_valid_mae = mean_absolute_error(y_valid, linear_valid_pred)
print(f'Линейная модель: Valid MAE = {linear_valid_mae:.1f} евро')


На вход `predict` снова подана исходная таблица, а не масштабированный массив. Если масштабировать её вручную ещё раз, получится двойное преобразование.

**Мини-задание:** покажите средние через `linear_pipeline.named_steps['scale'].mean_`. Объясните, откуда они взялись.


In [ ]:
# Ваш код: linear_pipeline.named_steps['scale'].mean_


## 3. Ближайшие соседи: учимся на похожих автомобилях

`KNeighborsRegressor` ищет k близких **обучающих** строк и, по умолчанию, усредняет их цены. Близость определяется численными признаками, а не человеческим понятием похожей модели автомобиля.

Для признаков в разных единицах масштаб важен: большие численные значения пробега могут доминировать в расстояниях. Поэтому перед соседями используем scaler.

`n_neighbors` — гиперпараметр: число соседей задаёт человек до `fit`. Оно не вычисляется автоматически из цен.


### Один конкретный вариант: 10 соседей


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn10 = Pipeline([
    ('scale', StandardScaler()),
    ('model', KNeighborsRegressor(n_neighbors=10))
])
knn10.fit(X_train, y_train)
knn10_valid_pred = knn10.predict(X_valid)
print(f'10 соседей: Valid MAE = {mean_absolute_error(y_valid, knn10_valid_pred):.1f} евро')


Вы получили прогнозы на тех же валидационных объектах. Сравните с линейной моделью.

**Остановитесь:** если в `predict` не передаются цены, откуда соседи берут значения для среднего?


### Рассмотрим соседей одного автомобиля

Сначала Pipeline преобразует строку. Затем `kneighbors` показывает расстояния и **позиции** соседей внутри обучающей таблицы. Для этих позиций нужен `iloc`, а не `loc`.


In [ ]:
one_car = X_valid.iloc[[0]]
one_scaled = knn10.named_steps['scale'].transform(one_car)
distances, positions = knn10.named_steps['model'].kneighbors(one_scaled)
neighbors = X_train.iloc[positions[0]].copy()
neighbors['Price'] = y_train.iloc[positions[0]].to_numpy()
neighbors['Distance_in_feature_space'] = distances[0]
display(one_car)
display(neighbors)
print('Средняя цена соседей:', neighbors['Price'].mean())
print('Прогноз:', knn10.predict(one_car)[0])


Среднее и прогноз совпали, поскольку используются равные веса соседей. Расстояние на этой таблице безразмерное после стандартизации; оно не является пробегом в км.

**Мини-задание:** какие признаки похожи у соседей, а какие различаются? Почему отсутствие марки может мешать такому определению близости?


## 4. Дерево решений и случайный лес

### Дерево

Дерево делит обучающие строки условиями по входным признакам, например `Year <= ...` или `Distance <= ...`. В листе выдаёт постоянную оценку цены, для стандартного квадратичного критерия — среднюю цену обучающих строк листа.

Модель не спрашивает неизвестную цену при прогнозе. `Price` не должна быть входным признаком.

Глубина управляет числом последовательных разделений: глубокое дерево может очень хорошо подстроиться под обучение и хуже работать на новых строках. Масштабирование для этих разделений не требуется.


### Обучаем небольшое дерево

Начнём с глубины 3, чтобы можно было прочитать правила.


In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

small_tree = DecisionTreeRegressor(max_depth=3, random_state=42)
small_tree.fit(X_train, y_train)
small_tree_valid_pred = small_tree.predict(X_valid)
print(f'Дерево глубины 3: Valid MAE = {mean_absolute_error(y_valid, small_tree_valid_pred):.1f} евро')


Дерево использовало ту же обучающую часть и проверялось на той же валидации. Другой алгоритм не требует другого разделения.


### Посмотрим на первые уровни

Условие написано в верхней строке узла. `samples` — число обучающих объектов, `value` — средняя обучающая цена в узле. На рисунке показываем только первые уровни ради читаемости.


In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))
plot_tree(small_tree, feature_names=NUMERIC_FEATURES, max_depth=2,
          impurity=False, filled=True, rounded=True, fontsize=9, ax=ax)
ax.set_title('Первые уровни дерева: обучение')
plt.show()


Выберите путь от корня вниз и перескажите его как последовательность вопросов. Скрытая часть рисунка не означает отсутствие более глубоких узлов.


### Случайный лес

Лес усредняет прогнозы многих деревьев, построенных на разных случайных подвыборках. Случайность может участвовать и в выборе признаков; в регрессоре по умолчанию на каждом разделении рассматриваются все признаки. Это не правило «каждое дерево обязательно видит разные колонки».

`n_estimators=60` — число деревьев, `max_depth=10` — ограничение их глубины. `n_jobs=1` оставляет расчёт в одном вычислительном задании ради предсказуемой нагрузки в учебной среде.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor(
    n_estimators=60, max_depth=10, random_state=42, n_jobs=1
)
forest.fit(X_train, y_train)
forest_valid_pred = forest.predict(X_valid)
print(f'Лес: Valid MAE = {mean_absolute_error(y_valid, forest_valid_pred):.1f} евро')


Не делайте вывод «лес всегда лучше» до сравнения чисел. Усреднение снижает часть изменчивости деревьев, но не делает модель нечувствительной к плохим данным и не гарантирует отсутствие переобучения.


## 5. Отдельные шаги знакомы — теперь небольшая общая функция

До этого мы каждый раз явно выполняли `fit`, `predict` и MAE. Повторение можно сократить. Функция ниже делает **только** эти знакомые действия и возвращает словарь. Она не выбирает параметры и не обращается к тесту.

| Аргумент | Что передаём |
|---|---|
| `model` | Один объект модели или Pipeline |
| `name` | Имя для строки отчёта |
| `X_fit`, `y_fit` | Обучающие данные |
| `X_check`, `y_check` | Данные проверки |

Функция обучает переданный объект; после вызова он становится обученным.


In [ ]:
def evaluate_model(model, name, X_fit, y_fit, X_check, y_check):
    model.fit(X_fit, y_fit)
    train_pred = model.predict(X_fit)
    check_pred = model.predict(X_check)
    return {
        'Model': name,
        'Train_MAE': mean_absolute_error(y_fit, train_pred),
        'Valid_MAE': mean_absolute_error(y_check, check_pred),
        'Valid_RMSE': np.sqrt(mean_squared_error(y_check, check_pred))
    }


Прочитайте функцию по строкам: обучение → два прогноза → три метрики → словарь.

### Список кандидатов

Начнём с небольшого фиксированного набора, а не десятков вариантов. Для всех одна таблица признаков и одно разделение. Масштабирование находится внутри Pipeline только там, где оно нужно.


In [ ]:
candidates = {
    'Median': DummyRegressor(strategy='median'),
    'Linear': Pipeline([('scale', StandardScaler()), ('model', LinearRegression())]),
    'KNN_1': Pipeline([('scale', StandardScaler()), ('model', KNeighborsRegressor(n_neighbors=1))]),
    'KNN_10': Pipeline([('scale', StandardScaler()), ('model', KNeighborsRegressor(n_neighbors=10))]),
    'KNN_30': Pipeline([('scale', StandardScaler()), ('model', KNeighborsRegressor(n_neighbors=30))]),
    'Tree_3': DecisionTreeRegressor(max_depth=3, random_state=42),
    'Tree_10': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Tree_full': DecisionTreeRegressor(random_state=42),
    'Forest_10': RandomForestRegressor(n_estimators=60, max_depth=10, random_state=42, n_jobs=1)
}


### Выполним сравнение

`items()` даёт имя и модель. В цикле обучаем и проверяем каждый вариант, складывая словари в список. Потом превращаем список в таблицу.


In [ ]:
results = []
for name, model in candidates.items():
    result = evaluate_model(model, name, X_train, y_train, X_valid, y_valid)
    results.append(result)

comparison_06 = pd.DataFrame(results).sort_values('Valid_MAE')
display(comparison_06.round(1))


Сортировка по `Valid_MAE` поставила меньшие ошибки выше. Значения MAE и RMSE здесь в евро.

Нулевая Train MAE у одного соседа возможна потому, что обучающая строка сама является своим ближайшим соседом. Это не доказательство качества на новых объектах.

**Задание:** сравните `Tree_3`, `Tree_10` и `Tree_full`. Как изменились ошибки на обучении и валидации? Найдите пример, где меньшая Train MAE не даёт меньшую Valid MAE.


### Нарисуем зависимость ошибок от глубины

График показывает три заранее выбранных варианта. Нельзя читать его как доказательство оптимальности всех промежуточных глубин.


In [ ]:
tree_rows = pd.DataFrame(results).set_index('Model').loc[['Tree_3', 'Tree_10', 'Tree_full']]
ax = tree_rows[['Train_MAE', 'Valid_MAE']].plot(marker='o', figsize=(8, 4))
ax.set(xlabel='Вариант дерева', ylabel='MAE, евро', title='Гибкость дерева и ошибка')
ax.grid(alpha=.3)
plt.tight_layout()
plt.show()


**Мини-задание:** объясните, почему самая гибкая модель не обязана оказаться лучшей на валидации. Изменяя параметры после просмотра графика, вы используете валидацию для настройки; тест для этого не открывайте.


## 6. Кросс-валидация: несколько внутренних проверок

Одно разделение может оказаться удачным или неудачным. KFold делит доступную обучающую часть на k блоков. На каждом шаге один блок служит внутренней проверкой, остальные — обучением. Модель создаётся заново для каждого шага.

Для спокойного знакомства возьмём **3 блока** и сравним линейную модель с соседями. Внешние `valid` и `test` здесь не участвуют.

| Шаг | Обучение внутри X_train | Проверка внутри X_train |
|---|---|---|
| 1 | Блоки 2 и 3 | Блок 1 |
| 2 | Блоки 1 и 3 | Блок 2 |
| 3 | Блоки 1 и 2 | Блок 3 |

Среднее трёх MAE описывает этот протокол. Стандартное отклонение между блоками не является доверительным интервалом и не гарантирует будущую ошибку.


### Настроим разбиение и посмотрим размеры

`KFold` здесь возвращает позиции строк. `shuffle=True` перемешивает строки перед формированием блоков.


In [ ]:
from sklearn.model_selection import KFold, cross_validate

cv = KFold(n_splits=3, shuffle=True, random_state=42)
for number, (fit_positions, check_positions) in enumerate(cv.split(X_train), start=1):
    print(f'Блок {number}: обучение {len(fit_positions)}, проверка {len(check_positions)}')


Каждая строка `X_train` один раз становится проверочной внутри CV. Для временных данных потребовалось бы другое разбиение, но даты объявлений здесь неизвестны.


### Запустим cross_validate для одного Pipeline

`scoring='neg_mean_absolute_error'` использует отрицательную MAE, потому что универсальный интерфейс scikit-learn считает большие scores лучшими. Для отчёта меняем знак обратно.

Pipeline заново обучает scaler **внутри каждого блока**. Если масштабировать весь X_train заранее и затем запускать CV по готовому массиву, информация внутренних проверочных блоков попадёт в подготовку.


In [ ]:
cv_result = cross_validate(
    candidates['KNN_10'], X_train, y_train,
    cv=cv, scoring='neg_mean_absolute_error',
    return_train_score=True, n_jobs=1
)
fold_mae = -cv_result['test_score']
display(pd.DataFrame({'Fold': [1, 2, 3], 'MAE, евро': fold_mae}).round(1))
print(f'Средняя MAE: {fold_mae.mean():.1f}; SD по блокам: {fold_mae.std():.1f} евро')


Ключ `test_score` здесь означает **внутренний проверочный блок CV**, а не внешний `X_test`. Мы внешний тест всё ещё не использовали.

**Остановитесь:** почему scaler внутри Pipeline важнее, чем просто сокращение кода?


### Сравним CV двух вариантов

Линейная модель и соседи получают одинаковые блоки. В основном маршруте выбор ниже всё равно опирается на заранее выделенную валидацию; CV служит дополнительной проверкой устойчивости сравнения.


In [ ]:
cv_rows = []
for name in ['Linear', 'KNN_10']:
    r = cross_validate(candidates[name], X_train, y_train, cv=cv,
                       scoring='neg_mean_absolute_error', n_jobs=1)
    values = -r['test_score']
    cv_rows.append({'Model': name, 'CV_MAE': values.mean(), 'CV_SD': values.std()})
cv_summary = pd.DataFrame(cv_rows)
display(cv_summary.round(1))


Сравните CV с отдельной валидацией. Числа не обязаны совпадать: в CV меньше обучающих строк и другие проверочные объекты. Не выбирайте способ отчёта только потому, что он даёт более красивую ошибку.


## 7. Выбрали вариант — теперь одна итоговая проверка

Для основного маршрута критерий зафиксирован: минимальная MAE на валидации среди кандидатов раздела 5. Тест ещё не использовался.

После выбора можно заново обучить выбранный алгоритм на **train + valid**: настройки уже определены. Scaler при этом также учится заново на этой объединённой части. Затем один раз оцениваем внешний test.

Не возвращайтесь после просмотра тестовых ошибок к подбору параметров с целью уменьшить именно тестовую ошибку. Если это произошло, тест стал частью разработки и для честной итоговой оценки нужен новый независимый набор.


### Запоминаем имя выбранного варианта

`iloc[0]` берёт первую строку таблицы, отсортированной по Valid MAE.


In [ ]:
best_name = comparison_06.iloc[0]['Model']
print('Выбран вариант:', best_name)
print('Критерий выбора: минимальная Valid MAE')


Мы выбрали алгоритм и параметры, а не самый удачный прогноз отдельной машины. Критерий не меняем после тестовой оценки.


### Создаём свежий объект и обучаем на 80%

`clone` копирует параметры модели или Pipeline, но не сохраняет обученные коэффициенты. `concat` объединяет обучающую часть с валидационной.


In [ ]:
from sklearn.base import clone

X_development = pd.concat([X_train, X_valid])
y_development = pd.concat([y_train, y_valid])
final_model = clone(candidates[best_name])
final_model.fit(X_development, y_development)
print('Для итогового обучения:', X_development.shape)


Теперь модель обучена на 80% исходных строк. Внешний тест остаётся отдельно. Не сравнивайте её новую обучающую ошибку с предыдущей так, будто состав данных остался прежним.


### Внешний тест

Получаем прогнозы без известных цен. Затем используем `y_test` только для измерения результата. Простой ориентир также переобучаем на тех же 80%, чтобы сравнение было согласованным.


In [ ]:
final_test_pred = final_model.predict(X_test)
final_baseline = DummyRegressor(strategy='median')
final_baseline.fit(X_development, y_development)
baseline_test_pred = final_baseline.predict(X_test)

test_summary = pd.DataFrame([
    {'Model': best_name,
     'Test_MAE': mean_absolute_error(y_test, final_test_pred),
     'Test_RMSE': np.sqrt(mean_squared_error(y_test, final_test_pred)),
     'Test_R2': r2_score(y_test, final_test_pred)},
    {'Model': 'Median',
     'Test_MAE': mean_absolute_error(y_test, baseline_test_pred),
     'Test_RMSE': np.sqrt(mean_squared_error(y_test, baseline_test_pred)),
     'Test_R2': r2_score(y_test, baseline_test_pred)}
])
display(test_summary.round(3))


Это итог выбранного эксперимента на отложенных строках. Хорошая цифра не гарантирует качество на будущем рынке, другой стране или машинах вне учебного отбора.

Теперь разрешается посмотреть тестовые ошибки **для отчёта и постановки будущих вопросов**. Исправлять по ним модель в этом же эксперименте нельзя.


### Анализ ошибок без нового подбора

Создадим таблицу только внешнего теста. В отличие от исходного большого примера, не смешиваем обучающие прогнозы с проверочными.


In [ ]:
test_predictions = df.loc[X_test.index, ['Make', 'Model', 'Year', 'Distance', 'Engine', 'Price']].copy()
test_predictions['Prediction'] = final_test_pred
test_predictions['Error'] = test_predictions['Prediction'] - test_predictions['Price']
test_predictions['Abs_error'] = test_predictions['Error'].abs()
display(test_predictions.nlargest(10, 'Abs_error'))
error_by_year = test_predictions.groupby('Year').agg(
    n=('Abs_error', 'size'), MAE=('Abs_error', 'mean'), Bias=('Error', 'mean')
)
display(error_by_year.sort_values('MAE', ascending=False).head(10).round(1))


`MAE` группы — величина ошибок, `Bias` — среднее направление. Всегда приводите `n`: большая MAE на двух старых машинах не характеризует весь такой рынок. Ошибка прогноза не доказывает некорректность исходной записи.


### Итоговый график

По горизонтали известная цена, по вертикали прогноз. Выше диагонали — завышение, ниже — занижение.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test, final_test_pred, s=10, alpha=.25)
low = min(y_test.min(), final_test_pred.min())
high = max(y_test.max(), final_test_pred.max())
ax.plot([low, high], [low, high], 'r--')
ax.set(title=f'{best_name}: внешний тест', xlabel='Указанная цена, евро', ylabel='Прогноз, евро')
ax.grid(alpha=.3)
plt.show()


Опишите три наблюдения по графику и таблице. Не формулируйте вывод «нужна сложнее модель» только по разбросу: могут отсутствовать существенные признаки или присутствовать неоднородные объекты.


## 8. Сохранение и запуск на новой строке

Сохраним итоговую модель вместе со всеми её преобразованиями. `joblib` подходит для учебного сохранения Python-объекта. Версии библиотек и порядок признаков тоже запишем. Загружайте только свои или доверенные файлы модели.

Модель предназначена для тех же полей и единиц. «Новая строка» ниже — демонстрация интерфейса на уже известной строке теста; второй независимой проверки она не создаёт.


### Сохранение


In [ ]:
import joblib
import json

joblib.dump(final_model, 'cars_price_model.joblib')
metadata = {'selected_model': best_name, 'numeric_features': NUMERIC_FEATURES,
            'price_unit': 'EUR', 'sklearn_version': sklearn.__version__,
            'selection_rule': 'minimum validation MAE',
            'reference_year_for_km_year': 2022}
Path('cars_price_model_metadata.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
)
comparison_06.to_csv('lab06_validation_comparison.csv', index=False)
cv_summary.to_csv('lab06_cv_summary.csv', index=False)
test_summary.to_csv('lab06_test_summary.csv', index=False)
test_predictions.rename_axis('dataset_row_id').reset_index().to_csv('lab06_test_predictions.csv', index=False)


Файл модели содержит выбранный алгоритм; если это Pipeline, scaler тоже сохранён. Для воспроизведения нужно помнить и правила подготовки входных полей из работы 02.


### Загрузка и predict

Сохраняем имена колонок. Не используем список из четырёх чисел без пояснения их порядка.


In [ ]:
loaded_model = joblib.load('cars_price_model.joblib')
new_car = X_test.iloc[[0]].copy()
display(new_car)
print(f'Оценка цены: {loaded_model.predict(new_car)[0]:.1f} евро')


Для модели с Pipeline здесь подаются исходные единицы. Повторно масштабировать строку вручную не нужно.

**Что сдавать:** выполненный ноутбук; таблицы сравнения, CV и итогового теста; модель и файл её описания; краткий вывод с выбранным вариантом, критерием выбора, ошибкой в евро и ограничениями.

**Оценивание, 10 баллов:** Pipeline и объяснение — 2; сравнение моделей — 2; гиперпараметры и переобучение — 2; CV и разделение ролей частей — 2; итоговый отчёт и запуск — 2.

Работа сравнила модели на одинаковых данных и отделила подбор от итоговой проверки. Полученный файл позволяет повторить предсказание при тех же полях и единицах.


## 9. Дополнительно: текстовые категории, по одному шагу

Этот раздел — отдельное расширение на **train/valid**, а не улучшение по тесту. Он не меняет `final_model` и не пересчитывает тестовые метрики.

У марки нет естественного численного порядка. Кодирование BMW=1, Toyota=2, Volkswagen=3 создаёт для некоторых алгоритмов выдуманные отношения. One-hot кодирование создаёт отдельную колонку 0/1 для каждой категории.

Если хотите выбрать этот вариант итоговым, сначала включите его в заранее определённый протокол сравнения, затем выполните итоговую проверку один раз. После уже просмотренного теста такая новая модель требует нового итогового набора.


### Крошечный пример OneHotEncoder

В основной таблице много категорий, поэтому сначала рассмотрим две марки. `fit` запоминает категории, `transform` создаёт индикаторы.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

toy_categories = pd.DataFrame({'Make': ['Toyota', 'BMW', 'Toyota']})
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(toy_categories)
toy_encoded = encoder.transform(toy_categories)
display(pd.DataFrame(toy_encoded, columns=encoder.get_feature_names_out()))


Для каждой строки одна из колонок равна 1. Две Toyota получили одинаковый код. Цена не использовалась.

`sparse_output=False` здесь выбирает обычный массив ради просмотра. На большой таблице разреженный формат хранит главным образом ненулевые значения и экономит память.


### Неизвестная категория

`handle_unknown='ignore'` не останавливает прогноз при новой категории: для неё в соответствующем блоке получается строка из нулей. Это техническое правило, а не знание о новой марке.


In [ ]:
unknown_make = pd.DataFrame({'Make': ['UnknownBrand']})
print(encoder.transform(unknown_make))


Модель не выучила свойства UnknownBrand. Посмотрите, почему отсутствие ошибки выполнения не означает надёжность прогнозов вне обучающих категорий.


### Выбираем категориальные поля

Берём марку, модель, кузов и топливо. Если в 02 сохранена исходная марка, используем её вместо объединённой `Rare`. Все выбранные колонки доступны в исходном `cars_processed.csv`; при наличии пропусков нужно сначала определить способ их обработки.


In [ ]:
CATEGORY_FEATURES = ['Make', 'Model', 'Style', 'Fuel_type']
X_mixed = df[NUMERIC_FEATURES + CATEGORY_FEATURES].copy()
if 'Make_original' in df:
    X_mixed['Make'] = df['Make_original']
X_mixed_train = X_mixed.loc[X_train.index]
X_mixed_valid = X_mixed.loc[X_valid.index]
display(X_mixed_train.head(3))


Автомобили и ответы остались прежними; расширился набор входов. Среднюю цену марки и года не добавляем: вычисленная по всей таблице, она раскрывает ответы проверочных объектов.


### ColumnTransformer: свои действия для разных колонок

Тройка `(имя, преобразователь, список колонок)` задаёт одну ветку. Численные поля стандартизируем, категориальные кодируем. `remainder='drop'` исключает все прочие колонки.


In [ ]:
from sklearn.compose import ColumnTransformer

mixed_preprocessor = ColumnTransformer([
    ('numbers', StandardScaler(), NUMERIC_FEATURES),
    ('categories', OneHotEncoder(handle_unknown='ignore'), CATEGORY_FEATURES)
], remainder='drop')
mixed_preprocessor.fit(X_mixed_train)
mixed_sample = mixed_preprocessor.transform(X_mixed_train.iloc[:3])
print('Три строки после преобразования:', mixed_sample.shape)
print('Первые имена:', mixed_preprocessor.get_feature_names_out()[:12])


Колонок стало больше, поскольку одно поле модели автомобиля превратилось во множество индикаторов. Параметры вычислялись только по train.


### Pipeline для смешанной таблицы

Собираем препроцессор и ту же линейную регрессию. Pipeline заново обучит свои шаги; отдельно преобразованный `mixed_sample` ей не передаём.


In [ ]:
mixed_pipeline = Pipeline([
    ('prepare', mixed_preprocessor),
    ('model', LinearRegression())
])
mixed_pipeline.fit(X_mixed_train, y_train)
mixed_valid_pred = mixed_pipeline.predict(X_mixed_valid)
print(f'Численные + категории, Valid MAE: {mean_absolute_error(y_valid, mixed_valid_pred):.1f} евро')
print(f'Только численные, Valid MAE: {linear_valid_mae:.1f} евро')


Сравнение показывает эффект категориальных полей на этой валидации. Оно не гарантирует улучшения на неизвестных моделях автомобилей. Сохраните этот результат отдельно от уже завершённого тестового эксперимента.


## 10. Дополнительно: логарифм цены

Обычная линейная регрессия предсказывает цену напрямую. Можно обучать её на `log(Price)`, затем вернуть прогноз через `exp`. Эти операции требуют положительных цен.

Оцениваем MAE и RMSE **после возврата в евро**, чтобы сравнение с предыдущими моделями было в одинаковых единицах. Обратное преобразование логарифма не гарантирует несмещённую оценку средней цены в исходной шкале.

`TransformedTargetRegressor` хранит преобразование ответа вместе с моделью. Это не преобразование входных признаков.


### Соберём преобразование целевой переменной

`func=np.log` применяется при обучении, `inverse_func=np.exp` — после предсказания. Сохраняем новый объект под отдельным именем, чтобы не менять завершённый эксперимент.


In [ ]:
from sklearn.compose import TransformedTargetRegressor

log_price_model = TransformedTargetRegressor(
    regressor=Pipeline([('scale', StandardScaler()), ('model', LinearRegression())]),
    func=np.log, inverse_func=np.exp
)
log_price_model.fit(X_train, y_train)
log_valid_pred = log_price_model.predict(X_valid)
print(f'Логарифм цены, Valid MAE: {mean_absolute_error(y_valid, log_valid_pred):.1f} евро')
print(f'Логарифм цены, Valid RMSE: {np.sqrt(mean_squared_error(y_valid, log_valid_pred)):.1f} евро')


`predict` уже возвращает цены в евро. Повторный `exp` был бы ошибкой. Нельзя смешивать метрики по логарифмам с метриками по ценам.


## 11. Дополнительно: полиномиальные признаки

Вместо смены алгоритма можно расширить представление входов. `PolynomialFeatures(degree=2)` добавляет квадраты и попарные произведения. Модель всё ещё линейна по **новым** признакам, но может описывать кривую связь с исходными.

Начинаем со степени 2, не со степени 3 и десятков вариантов. Расширение повышает гибкость и может ухудшить проверочный результат. `include_bias=False` не добавляет столбец единиц: свободный член уже есть в `LinearRegression`.


### Преобразование в одной последовательности

Сначала стандартизуем исходные величины, затем создаём степени и взаимодействия. Это уменьшает величины промежуточных чисел, но не устраняет все вопросы численной устойчивости.


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

polynomial_model = Pipeline([
    ('scale', StandardScaler()),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('model', LinearRegression())
])
polynomial_model.fit(X_train, y_train)
polynomial_valid_pred = polynomial_model.predict(X_valid)
print('Число новых входов:', polynomial_model.named_steps['poly'].n_output_features_)
print(f'Полином, Valid MAE: {mean_absolute_error(y_valid, polynomial_valid_pred):.1f} евро')


Здесь преобразуются признаки, а цена остаётся в евро. Это другая операция, чем логарифм целевой переменной.

**Задание:** сравните линейную, логарифмическую и полиномиальную версии на той же валидации. Сформулируйте вывод по фактическим числам без обещания, что преобразование всегда улучшает модель.


## 12. Дополнительно: тот же интерфейс в классификации

Короткий независимый пример на встроенном Iris показывает перенос интерфейса `fit`/`predict`. Цель здесь — класс цветка, а не цена. `LogisticRegression` вопреки названию является классификатором.

Поля измерений известны, целевой класс имеет три значения. `stratify=iris_y` сохраняет доли классов при разделении; это не операция для нашей непрерывной цены. Accuracy — доля верных классов, не метрика регрессионной цены.


In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

iris_X, iris_y = load_iris(return_X_y=True, as_frame=True)
iris_X_train, iris_X_test, iris_y_train, iris_y_test = train_test_split(
    iris_X, iris_y, test_size=.3, random_state=42, stratify=iris_y
)
iris_model = Pipeline([('scale', StandardScaler()),
                       ('model', LogisticRegression(max_iter=1000))])
iris_model.fit(iris_X_train, iris_y_train)
iris_pred = iris_model.predict(iris_X_test)
print(f'Доля правильных классов Iris: {accuracy_score(iris_y_test, iris_pred):.3f}')


Все переменные Iris названы отдельно: пример не меняет данные и модели автомобильного эксперимента. Не переносите accuracy на задачу цены.

### Дополнительный интерактивный просмотр тестовых ошибок

График ниже только показывает результаты уже выбранной модели. Он не является частью подбора. Наведите на точку, найдите строку и сформулируйте вопрос для будущего исследования.


In [ ]:
# Требуется Plotly: pip install plotly
import plotly.express as px

interactive_errors = test_predictions.rename_axis('dataset_row_id').reset_index()
error_fig = px.scatter(interactive_errors, x='Price', y='Prediction',
    hover_data=['dataset_row_id', 'Make', 'Model', 'Year', 'Distance', 'Error', 'Abs_error'],
    labels={'Price': 'Указанная цена, евро', 'Prediction': 'Прогноз, евро'},
    title='Внешний тест: просмотр ошибок итоговой модели', opacity=.5)
low = min(interactive_errors['Price'].min(), interactive_errors['Prediction'].min())
high = max(interactive_errors['Price'].max(), interactive_errors['Prediction'].max())
error_fig.add_shape(type='line', x0=low, y0=low, x1=high, y1=high,
                    line={'color': 'red', 'dash': 'dash'})
error_fig.show()
error_fig.write_html('lab06_test_errors.html', include_plotlyjs=True)


## Документация

- [Pipeline и составные преобразования](https://scikit-learn.org/stable/modules/compose.html)
- [Типичные ошибки и утечка данных](https://scikit-learn.org/stable/common_pitfalls.html)
- [Кросс-валидация](https://scikit-learn.org/stable/modules/cross_validation.html)
- [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html)
- [Метрики](https://scikit-learn.org/stable/modules/model_evaluation.html)
